In [1]:
from spark_utils import get_spark_session, stop_spark_session

# Import the types needed to build explicit schemas for each CSV
from pyspark.sql.types import (
    StructType,      # container for a table's column definitions
    StructField,     # a single column: name, type, nullability
    StringType,      # text columns
    IntegerType,     # whole-number columns
    DoubleType,      # decimal/float columns
    TimestampType,   # date+time columns
)

# Start/reuse a Spark session tagged specifically for this notebook's job
spark = get_spark_session("BronzeIngestion")

# Assigning the path to a variable where the Data files are located relative to notebooks/
RAW_DATA_DIR = "../data/raw"

# Assigning the path to a variable where Bronze Delta tables will be written relative to notebooks/
BRONZE_DIR = "../delta/bronze"

D:\Udemy_Python\envs\pyspark-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
# Explicit schema for each of the 9 Olist CSVs.
# Defining these by hand (instead of inferSchema=True) avoids Spark doing an extra full-file scan just to guess types, 
# and avoids type-guessing mistakes (e.g. turning zip-code prefixes into integers and dropping meaning).

schemas = {

    # Customers: dimtable
    "olist_customers_dataset": StructType([
        StructField("customer_id", StringType(), False),             # unique id per order's customer record
        StructField("customer_unique_id", StringType(), True),        # id that identifies the same real person across orders
        StructField("customer_zip_code_prefix", StringType(), True),    # first digits of zip code (kept as string to preserve leading zeros)
        StructField("customer_city", StringType(), True),
        StructField("customer_state", StringType(), True),
    ]),

    # Geolocation: maps Brazilian zip code prefixes to lat/lng coordinates (dimtable)
    "olist_geolocation_dataset": StructType([
        StructField("geolocation_zip_code_prefix", StringType(), True),  # zip code prefix, joins to customers/sellers
        StructField("geolocation_lat", DoubleType(), True),              # latitude
        StructField("geolocation_lng", DoubleType(), True),              # longitude
        StructField("geolocation_city", StringType(), True),
        StructField("geolocation_state", StringType(), True),
    ]),

    # Order items: dimtable
    "olist_order_items_dataset": StructType([
        StructField("order_id", StringType(), False),                 # links to orders table
        StructField("order_item_id", IntegerType(), False),
        StructField("product_id", StringType(), True),                # links to products table
        StructField("seller_id", StringType(), True),                 # links to sellers table
        StructField("shipping_limit_date", TimestampType(), True),    # seller's deadline to hand item to logistics partner
        StructField("price", DoubleType(), True),
        StructField("freight_value", DoubleType(), True),             # shipping cost for this item
    ]),

    # Payments: dimtable
    "olist_order_payments_dataset": StructType([
        StructField("order_id", StringType(), False),                 # links to orders table
        StructField("payment_sequential", IntegerType(), True),       # sequence if multiple payment methods used
        StructField("payment_type", StringType(), True),
        StructField("payment_installments", IntegerType(), True),     # number of installments chosen
        StructField("payment_value", DoubleType(), True),
    ]),

    # Reviews: dimtable
    "olist_order_reviews_dataset": StructType([
        StructField("review_id", StringType(), False),
        StructField("order_id", StringType(), True),                          # links to orders table
        StructField("review_score", IntegerType(), True),                     # out of 5
        StructField("review_comment_title", StringType(), True),              # may be null
        StructField("review_comment_message", StringType(), True),            # may be null
        StructField("review_creation_date", TimestampType(), True),           # when the review request was sent
        StructField("review_answer_timestamp", TimestampType(), True),        # when the customer submitted the review
    ]),

    # Orders: fact table
    "olist_orders_dataset": StructType([
        StructField("order_id", StringType(), False),                              # primary key
        StructField("customer_id", StringType(), False),                           # links to customers table
        StructField("order_status", StringType(), True),
        StructField("order_purchase_timestamp", TimestampType(), True),
        StructField("order_approved_at", TimestampType(), True),                   # when payment was approved
        StructField("order_delivered_carrier_date", TimestampType(), True),        # when handed to logistics partner
        StructField("order_delivered_customer_date", TimestampType(), True),       # actual delivery date to customer
        StructField("order_estimated_delivery_date", TimestampType(), True),       # estimated delivery date
    ]),

    # Products dimtable
    "olist_products_dataset": StructType([
        StructField("product_id", StringType(), False),                       # primary key
        StructField("product_category_name", StringType(), True),             # in Portuguese (translated later in Silver)
        StructField("product_name_lenght", IntegerType(), True),
        StructField("product_description_lenght", IntegerType(), True),
        StructField("product_photos_qty", IntegerType(), True),               # number of photos listed
        StructField("product_weight_g", DoubleType(), True),
        StructField("product_length_cm", DoubleType(), True),
        StructField("product_height_cm", DoubleType(), True),
        StructField("product_width_cm", DoubleType(), True),
    ]),

    # Sellers: dimtable
    "olist_sellers_dataset": StructType([
        StructField("seller_id", StringType(), False),
        StructField("seller_zip_code_prefix", StringType(), True),     # string to preserve leading zeros
        StructField("seller_city", StringType(), True),
        StructField("seller_state", StringType(), True),
    ]),

    # Category translation: dimtable
    "product_category_name_translation": StructType([
        StructField("product_category_name", StringType(), True),            # joins to products table
        StructField("product_category_name_english", StringType(), True),
    ]),
}

# Quick sanity check: confirm we defined a schema for all 9 expected tables
print(f"Schemas defined for {len(schemas)} tables:")
for table_name in schemas:
    print(f"  - {table_name}")

Schemas defined for 9 tables:
  - olist_customers_dataset
  - olist_geolocation_dataset
  - olist_order_items_dataset
  - olist_order_payments_dataset
  - olist_order_reviews_dataset
  - olist_orders_dataset
  - olist_products_dataset
  - olist_sellers_dataset
  - product_category_name_translation


In [3]:
# Ingest every table: read the raw CSV with its explicit schema, then write it as-is into the Bronze Delta layer.

for table_name, schema in schemas.items():

    csv_path = f"{RAW_DATA_DIR}/{table_name}.csv"     # path to this table's raw CSV file
    bronze_path = f"{BRONZE_DIR}/{table_name}"        # path where this table's Bronze Delta table will live

    # Read the CSV, enforcing our explicit schema instead of letting Spark guess types
    df = (
        spark.read
        .option("header", True)      # first row of the CSV has column names
        .option("quote", '"')        # handles field values that contain commas, wrapped in quotes
        .option("escape", '"')       # handles escaped quote characters inside quoted fields
        .schema(schema)              # apply our explicit schema instead of inferSchema for solid reliability
        .csv(csv_path)
    )

    row_count = df.count()   # how many rows landed

    # Write the raw (uncleaned) DataFrame out as a Delta table — this is the Bronze layer
    (
        df.write
        .format("delta")             # store in Delta Lake format, not plain CSV/Parquet
        .mode("overwrite")           # replace the table if this notebook is re-run
        .save(bronze_path)
    )

    print(f"[Bronze] {table_name}: {row_count} rows written to {bronze_path}")

print("\nBronze ingestion complete for all tables.")


[Bronze] olist_customers_dataset: 99441 rows written to ../delta/bronze/olist_customers_dataset
[Bronze] olist_geolocation_dataset: 1000163 rows written to ../delta/bronze/olist_geolocation_dataset
[Bronze] olist_order_items_dataset: 112650 rows written to ../delta/bronze/olist_order_items_dataset
[Bronze] olist_order_payments_dataset: 103886 rows written to ../delta/bronze/olist_order_payments_dataset
[Bronze] olist_order_reviews_dataset: 104162 rows written to ../delta/bronze/olist_order_reviews_dataset
[Bronze] olist_orders_dataset: 99441 rows written to ../delta/bronze/olist_orders_dataset
[Bronze] olist_products_dataset: 32951 rows written to ../delta/bronze/olist_products_dataset
[Bronze] olist_sellers_dataset: 3095 rows written to ../delta/bronze/olist_sellers_dataset
[Bronze] product_category_name_translation: 71 rows written to ../delta/bronze/product_category_name_translation

Bronze ingestion complete for all tables.


In [4]:
# Quick sanity check: read one Bronze table back from Delta and inspect it, to confirm the write actually worked

sample_df = spark.read.format("delta").load(f"{BRONZE_DIR}/olist_orders_dataset")  # read the orders Bronze table back

sample_df.printSchema()   # confirm the schema matches as defined
sample_df.show(5, truncate=False)   # preview the first 5 rows


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------

In [5]:
# Shut down the Spark session cleanly
stop_spark_session(spark)